In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from ml.data import load_data, split_data
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

In [ ]:
df = load_data()

X_train, X_test, y_train, y_test = split_data(df)
X_train.shape, X_test.shape

((1069, 6), (268, 6))

In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)

y_pred[:5]

array([13030.20336929, 13030.20336929, 13030.20336929, 13030.20336929,
       13030.20336929])

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mae, rmse, r2

(9861.797880026175, 13612.426003009037, -0.008391982350864469)

## Baseline

The `DummyRegressor` ignores all features and always predicts the **mean of the training target** (≈ 13,030) for every person.

| Metric | Value  | 
|--------|--------|
| MAE    | 9,862  |
| RMSE   | 13,612 |
| R²     | -0.008 |

- **MAE ≈ 9.9k**: guessing the mean, the error is about the size of a typical plan (median ≈ 9.4k).
- **RMSE is much higher than MAE** because a few huge errors dominate: smokers paying 40k+ still get the 13k guess.
- **R² ≈ 0** is expected: this model *is* "predict the mean". It is slightly negative because the training mean differs a bit from the test mean.

**This is the bar.** Any real model must reach an MAE below 9,862 and an R² above 0.


## Linear Regression

`LinearRegression` only works with numbers, so the data goes through a preprocessing step first:

- **Categorical** (`sex`, `smoker`, `region`) → `OneHotEncoder(drop="first")`: each category becomes a 0/1 column, dropping one redundant column per feature.
- **Numerical** (`age`, `bmi`, `children`) → `StandardScaler`: puts all columns on the same scale (mean 0, std 1).

Both steps are fitted **only on the training set** and wrapped with the model in a `Pipeline`, so the exact same transformation is applied to the test set and to new data (no leakage).


In [ ]:
numerical_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features),
    ]
)

preprocessor.fit(X_train)
preprocessor.get_feature_names_out()

array(['num__age', 'num__bmi', 'num__children', 'cat__sex_male',
       'cat__smoker_yes', 'cat__region_northwest',
       'cat__region_southeast', 'cat__region_southwest'], dtype=object)

In [ ]:
preprocessor.transform(X_test.head(1)).shape

(1, 8)

In [ ]:
lr_pipe = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

In [ ]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

mae_lr, rmse_lr, r2_lr

(4177.045561036326, 5956.342894363587, 0.8069287081198012)

In [ ]:
result = X_test.copy()

result["real"] = y_test
result["predicted"] = y_pred_lr
result["residual"] = result["real"] - result["predicted"]
result["obese"] = result["bmi"] >= 30

result.groupby(["smoker", "obese"])["residual"].mean()

smoker  obese
no      False    1371.561719
        True    -1750.799195
yes     False   -7463.443177
        True     9261.876105
Name: residual, dtype: float64

### Linear Regression results

| Metric | Dummy | Linear |
|---|---|---|
| MAE | 9,862 | 4,177 |
| RMSE | 13,612 | 5,956 |
| R² | -0.008 | 0.807 |

The linear model beats the baseline by far: the MAE drops by more than half and it explains ~81% of the variance.

**Where it fails** (mean residual = real − predicted, by group):
- **Non-smokers, not obese:** ≈ 1,371.56 **below** the real value (underestimates)
- **Non-smokers, obese:** ≈ 1,750.80 **above** the real value (overestimates)
- **Smokers, not obese:** ≈ 7,463.44 **above** the real value (overestimates)
- **Smokers, obese:** ≈ 9,261.88 **below** the real value (underestimates)

**Why:** the model learns a single weight for `smoker`, the same for every smoker. That weight ends up being an average of the two smoker groups: too small for obese smokers and too large for non-obese ones. It cannot represent the **smoker × BMI interaction** found in the EDA.


In [ ]:
coeficients = pd.Series(
    lr_pipe.named_steps["regressor"].coef_,
    index=lr_pipe.named_steps["preprocessor"].get_feature_names_out(),
)

coeficients.sort_values()

cat__region_southeast     -838.919616
cat__region_southwest     -659.139752
cat__region_northwest     -391.761455
cat__sex_male             -101.542054
num__children              636.501185
num__bmi                  1927.828251
num__age                  3472.975553
cat__smoker_yes          23077.764593
dtype: float64

### Coefficients

- **`smoker_yes` (+23k)** is by far the strongest factor: a smoker pays ~23k more than a non-smoker, all else equal.
- **`age` (+3.5k)** and **`bmi` (+1.9k)** come next. Since they were scaled, each value means "per 1 standard deviation" (~14 years of age, ~6 BMI points), which makes them comparable.
- **`sex`, `region` and `children`** barely change the price (hundreds).

The `smoker` weight is the same for every smoker, which is exactly why the model underestimates obese smokers and overestimates non-obese ones.
